In [ ]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SPLIT      = 'test'

FIELDVARS    = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS      = MODELS['nn']['seeds']
OPTIMIZEDEQS = MODELS['sr']['optimizedeqs']
PODRUNS      = MODELS['pod']['runs']
NNRUNS       = MODELS['nn']['runs']
ORDER        = ['pod_bl','nn_bl','nn_full','nn_nonparam','nn_gauss','sr_lo','sr_bl','sr_med','sr_hi']

COLORS = {}
LABELS = {}
for name,config in {**PODRUNS,**NNRUNS,**OPTIMIZEDEQS}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']

SRFUNCTIONS = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'neg':lambda x:-x,
    'sqrt':np.sqrt,
    'exp':np.exp,
    'log':np.log,
    'abs':np.abs,
    'max':np.maximum,
    'min':np.minimum}

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None:
        w = w*mask[:,None,:]
    return w.sum(axis=2)

def physical_prediction(output):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(output,dtype=float)))

def eval_form(form,variables,constants):
    ns = dict(SRFUNCTIONS)
    ns.update(variables)
    ns.update(constants)
    return np.asarray(eval(form,{'__builtins__':{}},ns),dtype=float)

def used_predictors(form,candidates):
    names = {n.id for n in ast.walk(ast.parse(form,mode='eval')) if isinstance(n,ast.Name)}
    return [c for c in candidates if c in names]

def r2(obs,pred):
    mask = np.isfinite(obs)&np.isfinite(pred)
    o,p  = obs[mask],pred[mask]
    return 1-np.sum((o-p)**2)/np.sum((o-o.mean())**2)

def pearsonr(x,y):
    mask = np.isfinite(x)&np.isfinite(y)
    x,y  = x[mask],y[mask]
    xc,yc = x-x.mean(),y-y.mean()
    denom = np.sqrt((xc**2).sum()*(yc**2).sum())
    return float((xc*yc).sum()/denom) if denom>0 else 0.0

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpmean = float(STATS['tp_mean'])
tpstd  = float(STATS['tp_std'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nlat  = ds.sizes['lat']
    nlon  = ds.sizes['lon']
    nsig  = ds.sizes.get('sig',1)
    lat   = ds['lat'].values
    lon   = ds['lon'].values
    dsig  = ds['dsig'].values
    farrs      = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS]
    fieldstack = np.stack(farrs,axis=1)
    surfmask   = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                  if 'surfmask' in ds else None)
    def getflat(da):
        if 'time' in da.dims:
            return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values,(ntime,1,1)).ravel()
    blnorm  = getflat(ds['bl'])
    lfnorm  = getflat(ds['lf'])
    shfnorm = getflat(ds['shf'])
    lhfnorm = getflat(ds['lhf'])
    sdonorm = getflat(ds['sdo'])
    # surface enthalpy flux and SST if present in splits
    sefnorm = getflat(ds['sef']) if 'sef' in ds else shfnorm + lhfnorm
    sstnorm = getflat(ds['sst']) if 'sst' in ds else None

kwlist = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            kwlist.append(wds['k'].values)
ki = np.mean([kernel_integrate(fieldstack,kw,dsig,surfmask) for kw in kwlist],axis=0) if kwlist else fieldstack.mean(axis=2)
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat  = ds.tp.transpose('time','lat','lon').values.ravel()
    # obs in normalized log-precip space
    obsnorm  = (np.log1p(obsflat)-tpmean)/tpstd

with open(os.path.join(MODELSDIR,'sr','optimized_equations.pkl'),'rb') as f:
    SR_REGISTRY = pickle.load(f)

print(f'Loaded {ntime} time steps, {nlat}×{nlon} grid')

In [ ]:
VARS = {'bl':blnorm,'rh':rhk,'thetae':thetaek,'thetaestar':thetaestark,
        'lf':lfnorm,'shf':shfnorm,'lhf':lhfnorm}

MODELPRED    = {}   # physical predictions
MODELRRATED  = {}   # raw form output (pre-physical_prediction) for SR equations

for name,cfg in OPTIMIZEDEQS.items():
    entry          = SR_REGISTRY.get(name,{})
    form           = entry.get('form',cfg['form'])
    constants      = entry.get('constants',cfg['init'])
    predictornames = used_predictors(form,VARS.keys())
    raw            = eval_form(form,{p:VARS[p] for p in predictornames},constants)
    MODELPRED[name]   = physical_prediction(raw)
    MODELRRATED[name] = raw   # keep raw form output for residual computation

SKIP = {'sr_rot'}

def load_predictions(name,split=SPLIT):
    path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    pred = da.transpose('time','lat','lon').values.ravel()
    return pred if pred.shape[0]==obsflat.shape[0] else None

for name,cfg in {**PODRUNS,**NNRUNS}.items():
    if name in SKIP: continue
    pred = load_predictions(name)
    if pred is not None: MODELPRED[name] = pred

valid = np.isfinite(obsflat)
print('Loaded predictions:', sorted(MODELPRED.keys()))
for name,pred in MODELPRED.items():
    print(f'  {LABELS.get(name,name):25s}  R²={r2(obsflat[valid],pred[valid]):.3f}')

## 1. Residuals: SR-HI vs. observations and vs. NN-GAUSS

Define residuals in three spaces:
- **Physical** (mm): `obs − SR-HI` and `NN-GAUSS − SR-HI`
- **Normalized log-precip**: `obs_norm − max(0, SR-HI_raw)` — this is exactly the additive correction a boosting SR run would need to learn
- **Log-space additive** (boss suggestion 2b): `log1p(obs) − log1p(SR-HI_phys)` — equivalent in the log-precip encoding

In [ ]:
sr_hi_phys = MODELPRED.get('sr_hi')
sr_hi_raw  = MODELPRED.get('sr_hi')   # placeholder if no raw stored
if 'sr_hi' in MODELPRED:
    entry      = SR_REGISTRY.get('sr_hi',{})
    cfg        = OPTIMIZEDEQS['sr_hi']
    form       = entry.get('form',cfg['form'])
    constants  = entry.get('constants',cfg['init'])
    pnames     = used_predictors(form,VARS.keys())
    sr_hi_raw  = eval_form(form,{p:VARS[p] for p in pnames},constants)  # raw equation output
    sr_hi_phys = physical_prediction(sr_hi_raw)

# Residuals in physical space
resid_phys    = obsflat - sr_hi_phys               # obs - SR-HI (mm)
gap_phys      = (MODELPRED['nn_gauss'] - sr_hi_phys
                 if 'nn_gauss' in MODELPRED else None)

# Additive residual in normalized log-precip space (target for a boosting SR)
resid_norm    = obsnorm - np.maximum(0.0, sr_hi_raw)   # what SR-HI misses, normalized

# Log-space additive residual (boss's equivalent framing)
resid_log     = np.log1p(np.maximum(0,obsflat)) - np.log1p(np.maximum(0,sr_hi_phys))

# Fractional (multiplicative) residual in physical space — only over rainy samples
with np.errstate(divide='ignore',invalid='ignore'):
    resid_mult = np.where(sr_hi_phys>0.01, obsflat/sr_hi_phys, np.nan)

print('Additive residual (norm) stats:')
print(f'  mean={np.nanmean(resid_norm[valid]):.4f}  std={np.nanstd(resid_norm[valid]):.4f}')
print(f'  range=[{np.nanpercentile(resid_norm[valid],1):.3f}, {np.nanpercentile(resid_norm[valid],99):.3f}]')
if gap_phys is not None:
    print(f'NN-GAUSS − SR-HI gap (physical, mm): mean={np.nanmean(gap_phys[valid]):.4f}  std={np.nanstd(gap_phys[valid]):.4f}')

## 2. Feature correlation ranking (boss suggestion 1)

Rank Pearson correlations of every available feature with:
- `obs − SR-HI` (physical): what is SR-HI systematically missing in the observations?
- `NN-GAUSS − SR-HI` (physical): what features explain what NN captures that SR does not?

High correlation → that feature is a promising addition to the SR equation or a new SR run targeting the residual.

In [ ]:
FEATURE_ARRAYS = {
    'KI(rh)':        rhk,
    'KI(θₑ)':        thetaek,
    'KI(θₑ*)':       thetaestark,
    'BL':            blnorm,
    'LF':            lfnorm,
    'SHF':           shfnorm,
    'LHF':           lhfnorm,
    'SEF (SHF+LHF)': sefnorm,
    'SDO':           sdonorm,
}
if sstnorm is not None:
    FEATURE_ARRAYS['SST'] = sstnorm

# Bowen ratio (SHF/LHF) — proxy for land surface state
with np.errstate(divide='ignore',invalid='ignore'):
    bow = np.where(np.abs(lhfnorm)>0.01, shfnorm/lhfnorm, np.nan)
FEATURE_ARRAYS['Bowen (SHF/LHF)'] = bow

resid_targets = {'obs − SR-HI': resid_phys}
if gap_phys is not None:
    resid_targets['NN-GAUSS − SR-HI'] = gap_phys

v = valid
corrs = {}
for fname, farr in FEATURE_ARRAYS.items():
    corrs[fname] = {tname: pearsonr(farr[v], tres[v])
                    for tname,tres in resid_targets.items()}

# Print ranked table
for tname in resid_targets:
    ranked = sorted(corrs.items(), key=lambda x: abs(x[1][tname]), reverse=True)
    print(f'\nCorrelations with  "{tname}" (ranked by |r|):')
    for fname,(rvals) in ranked:
        print(f'  {fname:22s}  r = {rvals[tname]:+.4f}')

In [ ]:
def plot_corr_bars():
    tnames = list(resid_targets.keys())
    ncols  = len(tnames)
    fig, axs = pplt.subplots(nrows=1, ncols=ncols, refwidth=3.2, refheight=3)
    axs = np.atleast_1d(axs).ravel()
    for ax, tname in zip(axs, tnames):
        fnames  = list(FEATURE_ARRAYS.keys())
        rvals   = [corrs[f][tname] for f in fnames]
        order   = np.argsort(rvals)     # ascending so most negative at bottom
        fnames  = [fnames[i] for i in order]
        rvals   = [rvals[i] for i in order]
        colors  = ['dodger blue' if r>0 else 'red' for r in rvals]
        ax.barh(np.arange(len(fnames)), rvals, color=colors, linewidth=0)
        ax.format(yticks=np.arange(len(fnames)), yticklabels=fnames,
                  xlabel='Pearson r', title=tname,
                  xlim=(-1,1), xlocator=[-0.5,0,0.5], grid=False)
        ax.axvline(0, color='gray', linewidth=0.7)
    pplt.show()

plot_corr_bars()

## 3. Spatial pattern of the additive residual

Time-mean `obs_norm − max(0, SR-HI_raw)` in normalized log-precip space — this is the signal a boosting SR model would need to reproduce. If it has clear geographic structure, spatial features (SDO, SE, LF) are the right candidates.

In [ ]:
import cartopy.crs as ccrs

def to_map(flat):
    return np.nanmean(flat.reshape(ntime,nlat,nlon),axis=0)

resid_norm_map = to_map(resid_norm)
resid_phys_map = to_map(resid_phys)
gap_map        = to_map(gap_phys) if gap_phys is not None else None

panels = [('obs − SR-HI\n(normalized log-precip)', resid_norm_map, 'DryWet', (-0.5,0.5), 'σ')]
panels.append(('obs − SR-HI\n(physical, mm)', resid_phys_map, 'DryWet', (-0.5,0.5), 'mm'))
if gap_map is not None:
    panels.append(('NN-GAUSS − SR-HI\n(physical, mm)', gap_map, 'DryWet', (-0.5,0.5), 'mm'))

fig, axs = pplt.subplots(nrows=1, ncols=len(panels), proj='cyl', refwidth=2.5, share=True)
axs = np.atleast_1d(axs).ravel()
for ax,(title,arr,cmap,(vmin,vmax),unit) in zip(axs,panels):
    m = ax.pcolormesh(lon, lat, arr, cmap=cmap, vmin=vmin, vmax=vmax,
                      extend='both', transform=ccrs.PlateCarree())
    ax.format(title=title, latlim=(lat.min(),lat.max()),
              lonlim=(lon.min(),lon.max()),
              coast=True, borders=False, grid=True, gridminor=False)
    fig.colorbar(m, loc='b', label=unit, ax=ax, width=0.15)
pplt.show()

## 4. Multiplicative residual in log-precip space (boss suggestion 2b)

In the log-precip encoding, the model target is `log1p(obs)`. An additive correction in log-precip space is a *multiplicative* correction in physical space (since `log(A·B) = log(A) + log(B)`). Plot the log-space residual `log1p(obs) − log1p(SR-HI)` as a function of intensity and of each feature to see whether a simple functional form can explain it.

In [ ]:
def bin_1d(x, z, nbins=20, minsamples=100, plo=1, phi=99):
    mask = np.isfinite(x)&np.isfinite(z)
    x, z = x[mask], z[mask]
    lo, hi = np.percentile(x, [plo, phi])
    edges  = np.linspace(lo, hi, nbins+1)
    idxs   = np.clip(np.digitize(x, edges)-1, 0, nbins-1)
    xc     = 0.5*(edges[:-1]+edges[1:])
    means  = np.full(nbins, np.nan)
    for i in range(nbins):
        idx = idxs==i
        if idx.sum()>=minsamples:
            means[i] = z[idx].mean()
    return xc, means

fig, axs = pplt.subplots(nrows=2, ncols=4, refwidth=2.2, refheight=1.8, share=False)
axs = axs.ravel()

plot_features = [
    ('KI(rh)', rhk),
    ('KI(θₑ)', thetaek),
    ('KI(θₑ*)', thetaestark),
    ('BL', blnorm),
    ('LF', lfnorm),
    ('SHF', shfnorm),
    ('SEF', sefnorm),
    ('SDO', sdonorm),
]

resid_log_v = resid_log[valid]
for ax,(fname,farr) in zip(axs, plot_features):
    xc, means = bin_1d(farr[valid], resid_log_v)
    ax.plot(xc, means, color='k', linewidth=1.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.format(xlabel=fname, ylabel='log1p residual', title='', grid=False)

fig.format(suptitle='Log-space residual (log1p(obs) − log1p(SR-HI)) vs. each feature')
pplt.show()

## 5. Save residual targets for SR boosting runs

Save the additive residual in normalized log-precip space as a NumPy array. This can be used as the target variable for a new PySR run that learns what SR-HI is missing. The residual `r = obs_norm − max(0, SR-HI_raw)` satisfies:

    obs_norm ≈ max(0, SR-HI_raw) + g(features)    [additive boosting]

or equivalently in log-precip space:

    log1p(obs) ≈ log1p(SR-HI) + h(features)        [log-space additive]

A PySR run on `r` with features `{rh, thetae, thetaestar, bl, lf, shf, lhf, sef, sdo}` will find the best symbolic correction.

Feature priority based on correlations above — look for features with the largest |r| with `obs − SR-HI` that are not already well-represented in SR-HI's form.

In [ ]:
# Save residual targets and feature matrix for a potential SR boosting run
outdir = os.path.join(MODELSDIR,'sr')
os.makedirs(outdir, exist_ok=True)

# Feature matrix: (nsamples, nfeatures) — only valid samples
feat_names_out = ['rh','thetae','thetaestar','bl','lf','shf','lhf','sef','sdo']
feat_arrays    = [rhk, thetaek, thetaestark, blnorm, lfnorm, shfnorm, lhfnorm, sefnorm, sdonorm]
X = np.stack([arr[valid] for arr in feat_arrays], axis=1)
y_add  = resid_norm[valid]    # additive normalized residual
y_log  = resid_log[valid]     # log-space additive residual

np.save(os.path.join(outdir,'resid_X.npy'), X.astype(np.float32))
np.save(os.path.join(outdir,'resid_y_additive_norm.npy'), y_add.astype(np.float32))
np.save(os.path.join(outdir,'resid_y_log.npy'), y_log.astype(np.float32))

with open(os.path.join(outdir,'resid_feature_names.json'),'w') as f:
    json.dump(feat_names_out, f)

print(f'Saved residual targets to {outdir}')
print(f'  X shape: {X.shape}  (features: {feat_names_out})')
print(f'  y_add range: [{y_add.min():.3f}, {y_add.max():.3f}]')
print(f'  y_log range: [{y_log.min():.3f}, {y_log.max():.3f}]')